In [4]:
import pandas as pd
import numpy as np
#import seaborn as sns
#import matplotlib.pyplot as plt
#rom scipy import stats

In [5]:
fl = pd.read_csv('cstJapa.csv')

# univariate gradient boosting 

In [36]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Create a 'Month' column based on the 'week_number'

fl['Month'] = (fl['Week number'] - 1) // 4 + 1

# Aggregate the total flu cases by year and month
monthly_flu = fl.groupby(['Year', 'Month'])['total_flu'].sum().reset_index()

# Create lag features to use previous months' flu cases to predict the next month
monthly_flu['lag_1'] = monthly_flu['total_flu'].shift(1)
monthly_flu['lag_2'] = monthly_flu['total_flu'].shift(2)
monthly_flu['lag_3'] = monthly_flu['total_flu'].shift(3)

# Drop rows with NaN values due to lagging
monthly_flu = monthly_flu.dropna()

# Define X (lag features) and y (current month's total flu)
X = monthly_flu[['lag_1', 'lag_2', 'lag_3']]
y = monthly_flu['total_flu']

# Train-test split (No shuffling as it's a time series)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Initialize Gradient Boosting Regressor with Huber loss
gb_model = GradientBoostingRegressor(loss='huber', random_state=42, n_iter_no_change=10, validation_fraction=0.2)

# Define the parameter distribution for RandomizedSearchCV
param_dist = {
    'n_estimators': [50, 100, 150],   # Reduce range of trees
    'learning_rate': [0.01, 0.05],    # Smaller learning rates
    'max_depth': [3, 4],              # Keep depth small
    'min_samples_split': [2, 5],      # Regularize
    'min_samples_leaf': [1, 2],       # Regularize
    'subsample': [0.8, 1.0]           # Add subsampling
}

# Use RandomizedSearchCV
random_search = RandomizedSearchCV(estimator=gb_model,
                                   param_distributions=param_dist,
                                   n_iter=30, 
                                   cv=5,  # Fewer folds due to small dataset
                                   verbose=2,
                                   random_state=42,
                                   n_jobs=-1)

# Fit the RandomizedSearchCV model
random_search.fit(X_train, y_train)

# Get the best estimator
best_gb_model = random_search.best_estimator_

# Make predictions on the test set
y_pred = best_gb_model.predict(X_test)

# Evaluate the model
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)

# Print the best parameters and evaluation metrics
print(f"Best parameters found: {random_search.best_params_}")
print(f"R-squared: {r2:.3f}")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")




Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters found: {'subsample': 1.0, 'n_estimators': 50, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 4, 'learning_rate': 0.05}
R-squared: 0.340
MSE: 1213770.14
RMSE: 1101.71
MAE: 520.92


# univariate random forest 

In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

'

# Step 1: Aggregate flu cases by year and month
monthly_flu = fl.groupby(['Year', 'Month'])['total_flu'].sum().reset_index()

# Step 2: Create lag features based on previous months' data
monthly_flu['lag_1'] = monthly_flu['total_flu'].shift(1)
monthly_flu['lag_2'] = monthly_flu['total_flu'].shift(2)
monthly_flu['lag_3'] = monthly_flu['total_flu'].shift(3)

# Drop rows with NaN values due to lagging
monthly_flu = monthly_flu.dropna()

# Step 3: Define X (lag features) and y (current month's total flu)
X = monthly_flu[['lag_1', 'lag_2', 'lag_3']].values  # Use the lag features as input
y = monthly_flu['total_flu'].values  # Target variable is the current month's flu cases

# Step 4: Split the data into training and testing sets (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False, random_state=42)

# Step 5: Define the Random Forest model
rf_model = RandomForestRegressor(random_state=42)

# Step 6: Define the parameter distribution for RandomizedSearchCV
param_dist = {
    'n_estimators': [100, 200, 300],  # Number of trees
    'max_depth': [5, 10, 15],  # Limit tree depth
    'min_samples_split': [10, 20, 30],  # Split to avoid overfitting
    'min_samples_leaf': [2, 5, 10],  # Leaf size to prevent overfitting
    'max_features': ['sqrt', 'log2'],  # Features considered at each split
    'bootstrap': [True, False]  # Try bootstrapping or not
}

# Step 7: Use RandomizedSearchCV for hyperparameter tuning
random_search = RandomizedSearchCV(estimator=rf_model, 
                                   param_distributions=param_dist, 
                                   n_iter=50,  # Number of random combinations to try
                                   cv=5,  # 5-fold cross-validation
                                   verbose=2, 
                                   random_state=42, 
                                   n_jobs=-1)  # Use all available processors

# Step 8: Fit the RandomizedSearchCV model
random_search.fit(X_train, y_train)

# Step 9: Get the best estimator from RandomizedSearchCV
best_rf_model = random_search.best_estimator_

# Step 10: Predict on the test set using the best model
y_pred_rf = best_rf_model.predict(X_test)

# Step 11: Calculate performance metrics
r2_rf = r2_score(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)

# Step 12: Print the best parameters and the results
print(f"Best parameters found: {random_search.best_params_}")
print(f"R-squared: {r2_rf:.3f}")
print(f"MSE: {mse_rf:.2f}")
print(f"RMSE: {rmse_rf:.2f}")
print(f"MAE: {mae_rf:.2f}")


Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best parameters found: {'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 15, 'bootstrap': False}
R-squared: 0.602
MSE: 731359.73
RMSE: 855.20
MAE: 507.51


# univariate lstm 

In [15]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.wrappers.scikit_learn import KerasRegressor
import math
import tensorflow as tf


fl['Month'] = (fl['Week number'] - 1) // 4 + 1

# Aggregate flu cases by year and month
monthly_flu = fl.groupby(['Year', 'Month'])['total_flu'].sum().reset_index()

# Create lag features to use previous months' flu cases for prediction
monthly_flu['lag_1'] = monthly_flu['total_flu'].shift(1)
monthly_flu['lag_2'] = monthly_flu['total_flu'].shift(2)
monthly_flu['lag_3'] = monthly_flu['total_flu'].shift(3)

# Drop rows with NaN values due to lagging
monthly_flu = monthly_flu.dropna()

# Define X (lag features) and y (current month's total flu)
X = monthly_flu[['lag_1', 'lag_2', 'lag_3']].values  # Use the lag features as input
y = monthly_flu['total_flu'].values  # Target variable is the current month's flu cases

# Split the data into training and testing sets (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features and target to the [0, 1] range (required for neural networks)
scaler_X = MinMaxScaler(feature_range=(0, 1))
scaler_Y = MinMaxScaler(feature_range=(0, 1))

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

y_train_scaled = scaler_Y.fit_transform(y_train.reshape(-1, 1))
y_test_scaled = scaler_Y.transform(y_test.reshape(-1, 1))

# Reshape input for LSTM (LSTM expects input as [samples, time steps, features])
X_train_scaled = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_test_scaled = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

# Function to create the LSTM model
def create_lstm_model(units=64, learning_rate=0.01):
    model = Sequential()
    model.add(LSTM(units=units, return_sequences=True, input_shape=(X_train_scaled.shape[1], X_train_scaled.shape[2])))
    model.add(LSTM(units=units))
    model.add(Dense(1))  # Output layer for regression

    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mean_squared_error')
    
    return model

# Wrap the LSTM model with KerasRegressor for RandomizedSearchCV
model = KerasRegressor(build_fn=create_lstm_model, verbose=0)

# Define the hyperparameter grid for RandomizedSearchCV
param_dist = {
    'batch_size': [16, 32],    # Try smaller batch sizes
    'epochs': [50, 100, 200],  # Different epoch options
    'learning_rate': [0.001, 0.01],  # Different learning rates
    'units': [64, 128]        # Different LSTM unit sizes
}

# Set up RandomizedSearchCV
random_search = RandomizedSearchCV(estimator=model, param_distributions=param_dist, n_iter=10, cv=3, random_state=42, n_jobs=1)

# Fit RandomizedSearchCV to the data
random_search.fit(X_train_scaled, y_train_scaled)

# Get the best estimator and print the best parameters
best_model = random_search.best_estimator_
best_params = random_search.best_params_
print(f"Best Parameters: {best_params}")

# Make predictions on the test set using the best model
y_pred_scaled = best_model.predict(X_test_scaled)

# Inverse transform the scaled predictions and actual values
y_pred = scaler_Y.inverse_transform(y_pred_scaled.reshape(-1, 1))
y_test_actual = scaler_Y.inverse_transform(y_test_scaled)

# Calculate evaluation metrics
mse = mean_squared_error(y_test_actual, y_pred)
mae = mean_absolute_error(y_test_actual, y_pred)
rmse = math.sqrt(mse)
r2 = r2_score(y_test_actual, y_pred)

# Create a DataFrame for the metrics
metrics_df = pd.DataFrame({
    'Metric': ['R-squared', 'MSE', 'MAE', 'RMSE'],
    'Value': [r2, mse, mae, rmse]
})

# Display the metrics DataFrame
metrics_df


C:\Users\User\AppData\Local\Temp\ipykernel_22672\2805695414.py:61: DeprecationWarning: KerasRegressor is deprecated, use Sci-Keras (https://github.com/adriangb/scikeras) instead. See https://www.adriangb.com/scikeras/stable/migration.html for help migrating.
  model = KerasRegressor(build_fn=create_lstm_model, verbose=0)


Best Parameters: {'units': 128, 'learning_rate': 0.001, 'epochs': 50, 'batch_size': 32}


,Metric,Value
0,R-squared,2.224950e-01
1,MSE,3.341446e+06
2,MAE,9.087645e+02
3,RMSE,1.827962e+03


In [16]:
print(f"R-squared: {r2:.3f}")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")

R-squared: 0.222
MSE: 3341445.90
RMSE: 1827.96
MAE: 908.76
